# Elementos de Redes I
## Fundamentos, representações e modelagem com NetworkX

**Disciplina:** Estrutura de Dados II — Grafos  
**Formato:** aula prática e investigativa em Python  
**Biblioteca principal:** [NetworkX](https://networkx.org/)

> Uma rede não é apenas um desenho: é uma decisão sobre quais objetos serão vértices, quais relações serão arestas e quais informações precisam ser preservadas.

## Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

1. modelar um problema como um grafo $G=(V,E)$ ou como um grafo ponderado $G=(V,E,w)$;
2. distinguir grafos não direcionados, direcionados, ponderados e multigrafos;
3. construir, consultar e modificar grafos com NetworkX;
4. interpretar ordem, tamanho, grau, vizinhança;
5. relacionar listas, dicionários e matrizes de adjacência ao custo de armazenamento;
6. verificar propriedades matemáticas por meio de código;
7. justificar escolhas de modelagem em aplicações reais.

### Roteiro sugerido

| Bloco | Tema | Tempo sugerido |
|---|---|---:|
| 1 | Motivação e modelo matemático | 15 min |
| 2 | Construção e inspeção com NetworkX | 25 min |
| 3 | Representações e complexidade | 20 min |
| 4 | Dígrafos, pesos e multigrafos | 25 min |
| 5 | Estudo de caso e exercícios | 30–45 min |



## 1. Por que grafos?

Muitos sistemas podem ser descritos por **entidades** e **relações**:

- pessoas e amizades;
- aeroportos e voos;
- páginas e hyperlinks;
- artigos e citações;
- componentes de software e dependências;
- conceitos e relações semânticas em grafos de conhecimento;
- configurações possíveis de um jogo e alterações entre configurações.

A abstração é poderosa porque os mesmos algoritmos podem ser reutilizados em domínios diferentes. O ponto crítico é a **semântica**: uma aresta pode significar amizade, distância, dependência, fluxo, similaridade ou outra relação.

### Pergunta para discussão

Em uma rede de voos, o que muda se o peso de uma aresta representar distância, tempo, preço ou número de passageiros?

## 2. Configuração do ambiente

O notebook usa apenas bibliotecas comuns no Jupyter e no Google Colab. Não há downloads de arquivos externos; os pequenos conjuntos de dados estão embutidos para garantir reprodutibilidade.

In [ ]:
from itertools import product
from math import comb
import platform

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 11,
})

print(f"Python: {platform.python_version()}")
print(f"NetworkX: {nx.__version__}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")

## 3. Modelo matemático e vocabulário

Um **grafo simples não direcionado** é um par:

$$G=(V,E),$$

em que $V$ é o conjunto de vértices (ou nós) e $E\subseteq\{\{u,v\}:u,v\in V, u\neq v\}$ é o conjunto de arestas.

Quando há um valor associado a cada aresta, usamos uma função de peso:

$$w:E\rightarrow\mathbb{R}, \qquad G=(V,E,w).$$

| Termo | Símbolo/descrição | Interpretação |
|---|---|---|
| ordem | $n=|V|$ | número de vértices |
| tamanho | $m=|E|$ | número de arestas |
| vizinhança | $N(v)$ | vértices adjacentes a $v$ |
| grau | $d(v)=|N(v)|$ | número de arestas incidentes em $v$ |
| caminho | sequência de vértices adjacentes | rota possível na rede |
| componente | subgrafo conectado maximal | região alcançável internamente |
| laço | aresta de um vértice para ele mesmo | permitido em alguns modelos |
| arestas paralelas | várias arestas entre o mesmo par | exige um multigrafo |

**Atenção:** o peso não significa necessariamente custo. Se o peso for força, capacidade ou similaridade, “menor peso” pode não representar o melhor caminho.

### Escolha da estrutura em NetworkX

| Classe | Direcionado? | Arestas paralelas? | Exemplo |
|---|:---:|:---:|---|
| `nx.Graph` | não | não | amizade, colaboração |
| `nx.DiGraph` | sim | não | seguidores, dependências |
| `nx.MultiGraph` | não | sim | várias linhas entre estações |
| `nx.MultiDiGraph` | sim | sim | múltiplos voos direcionados |

Todas aceitam atributos em vértices e arestas. Um atributo chamado `weight` é reconhecido por diversos algoritmos, mas outros nomes também podem ser utilizados quando informados explicitamente.

## 4. Primeiro grafo: construção, inspeção e desenho

In [ ]:
G = nx.Graph(name="Rede didática")

# Nós podem ser inseridos individualmente ou em lote.
G.add_node("A")
G.add_nodes_from(["B", "C", "D", "E"])

# Em um grafo não direcionado, (A, B) e (B, A) representam a mesma aresta.
G.add_edges_from([
    ("A", "B"),
    ("A", "C"),
    ("B", "C"),
    ("C", "D"),
])

print("Nós:", list(G.nodes))
print("Arestas:", list(G.edges))
print("Ordem n =", G.number_of_nodes())
print("Tamanho m =", G.number_of_edges())

O nó `E` foi adicionado, mas não possui arestas: ele é um **vértice isolado**. O desenho ajuda a comunicar a estrutura, mas posições geométricas não fazem parte do grafo, a menos que sejam dados explícitos do problema.

In [ ]:
def desenhar_grafo(
    grafo,
    *,
    pos=None,
    titulo=None,
    node_color="#2A6FBB",
    node_size=1600,
    font_size=10,
    edge_labels=None,
    arrows=None,
):
    # Desenha grafos pequenos de forma consistente ao longo da aula.
    if pos is None:
        pos = nx.spring_layout(grafo, seed=SEED)
    if arrows is None:
        arrows = grafo.is_directed()

    fig, ax = plt.subplots(figsize=(8, 5))
    nx.draw_networkx(
        grafo,
        pos,
        ax=ax,
        with_labels=True,
        node_color=node_color,
        node_size=node_size,
        font_color="white",
        font_size=font_size,
        font_weight="bold",
        edge_color="#667085",
        width=1.8,
        arrows=arrows,
        arrowsize=20,
    )
    if edge_labels:
        nx.draw_networkx_edge_labels(grafo, pos, edge_labels=edge_labels, ax=ax)
    ax.set_title(titulo or grafo.graph.get("name", "Grafo"))
    ax.axis("off")
    plt.show()
    return pos


pos_G = desenhar_grafo(G, titulo="Grafo simples com um vértice isolado")

### Views, iteradores e cópias

Em NetworkX, `G.nodes`, `G.edges` e `G.degree` são **views**: objetos leves que refletem alterações no grafo. Converta-os para `list` ou `dict` quando precisar de uma fotografia independente do estado atual.

In [ ]:
print("Vizinhos de C:", list(G.neighbors("C")))
print("Existe o nó A?", G.has_node("A"))
print("Existe a aresta A-D?", G.has_edge("A", "D"))
print("Graus:", dict(G.degree()))

graus_antes = dict(G.degree())
G.add_edge("D", "E")
print("Após inserir D-E:", dict(G.degree()))
print("Cópia anterior preservada:", graus_antes)

### Remoção e cópia defensiva

Operações como `add_edge`, `remove_node` e `remove_edge` alteram o grafo **in-place**. Ao experimentar uma modificação que não deve afetar o original, use `G.copy()`.

In [ ]:
G_experimento = G.copy()
G_experimento.remove_node("C")

print("Nós no original:", sorted(G.nodes))
print("Nós na cópia modificada:", sorted(G_experimento.nodes))

## 5. Representações: lista, dicionário e matriz de adjacência

Uma lista de adjacência armazena, para cada vértice, seus vizinhos. Para grafos esparsos, essa representação costuma exigir espaço $\Theta(|V|+|E|)$. Uma matriz de adjacência exige $\Theta(|V|^2)$ posições, mas oferece acesso direto a uma possível aresta.

| Operação típica | Lista/dicionário de adjacência | Matriz de adjacência |
|---|---:|---:|
| espaço | $\Theta(V+E)$ | $\Theta(V^2)$ |
| testar $(u,v)$ | depende da estrutura; em NetworkX, tipicamente $O(1)$ médio | $O(1)$ |
| percorrer vizinhos de $u$ | $\Theta(d(u))$ | $\Theta(V)$ |
| percorrer todas as arestas | $\Theta(V+E)$ | $\Theta(V^2)$ |

NetworkX usa internamente estruturas semelhantes a dicionários de dicionários, favorecendo grafos esparsos e atributos flexíveis.

In [ ]:
lista_adjacencia = {no: sorted(G.neighbors(no)) for no in G.nodes}
display(pd.Series(lista_adjacencia, name="vizinhos").to_frame())

ordem_nos = sorted(G.nodes)
matriz = nx.to_numpy_array(G, nodelist=ordem_nos, dtype=int)
display(pd.DataFrame(matriz, index=ordem_nos, columns=ordem_nos))

Para um grafo simples não direcionado, a matriz de adjacência é simétrica. Em um dígrafo, $A_{ij}=1$ registra a aresta $i\rightarrow j$ e a matriz não precisa ser simétrica.

In [ ]:
print("A matriz é simétrica?", np.array_equal(matriz, matriz.T))
print("Soma da matriz:", matriz.sum())
print("2m:", 2 * G.number_of_edges())

## 6. Atributos em nós e arestas

A estrutura topológica informa **quem está ligado a quem**. Atributos preservam informações do domínio, como tipo, cidade, capacidade, duração ou custo.

In [ ]:
SG = nx.Graph(name="Rede social da turma")
SG.add_nodes_from([
    ("Ana", {"curso": "Computação", "periodo": 3}),
    ("Bruno", {"curso": "Computação", "periodo": 5}),
    ("Carla", {"curso": "Engenharia", "periodo": 3}),
    ("Diego", {"curso": "Engenharia", "periodo": 7}),
    ("Eva", {"curso": "Computação", "periodo": 5}),
    ("Fábio", {"curso": "Engenharia", "periodo": 3}),
    ("Gabi", {"curso": "Computação", "periodo": 7}),
])

SG.add_edges_from([
    ("Ana", "Bruno", {"interacoes": 8}),
    ("Ana", "Carla", {"interacoes": 4}),
    ("Ana", "Fábio", {"interacoes": 7}),
    ("Bruno", "Carla", {"interacoes": 5}),
    ("Bruno", "Eva", {"interacoes": 9}),
    ("Carla", "Diego", {"interacoes": 6}),
    ("Carla", "Fábio", {"interacoes": 3}),
    ("Diego", "Fábio", {"interacoes": 6}),
    ("Diego", "Gabi", {"interacoes": 2}),
    ("Eva", "Fábio", {"interacoes": 5}),
    ("Fábio", "Gabi", {"interacoes": 8}),
])

cores_curso = {"Computação": "#2A6FBB", "Engenharia": "#E76F51"}
cores = [cores_curso[SG.nodes[n]["curso"]] for n in SG.nodes]
larguras = [0.6 + SG[u][v]["interacoes"] / 3 for u, v in SG.edges]
pos_SG = nx.spring_layout(SG, seed=SEED)

fig, ax = plt.subplots(figsize=(9, 6))
nx.draw_networkx(
    SG,
    pos_SG,
    ax=ax,
    node_color=cores,
    node_size=1300,
    font_color="white",
    font_weight="bold",
    width=larguras,
    edge_color="#98A2B3",
)
ax.set_title("Rede social: cor = curso; espessura = número de interações")
ax.axis("off")
plt.show()

In [ ]:
nos_df = pd.DataFrame.from_dict(dict(SG.nodes(data=True)), orient="index")
nos_df.index.name = "pessoa"

arestas_df = pd.DataFrame(
    [(u, v, dados["interacoes"]) for u, v, dados in SG.edges(data=True)],
    columns=["origem", "destino", "interacoes"],
)

display(nos_df)
display(arestas_df.sort_values("interacoes", ascending=False).head())

## 7. Grafos direcionados

Em um dígrafo, uma aresta é um par ordenado $(u,v)$ e representa $u\rightarrow v$. Assim, $(u,v)$ e $(v,u)$ são relações distintas.

Para cada vértice $v$:

- $d^-(v)$: grau de entrada;
- $d^+(v)$: grau de saída;
- $d(v)=d^-(v)+d^+(v)$ em NetworkX.

Além disso:

$$\sum_v d^-(v)=\sum_v d^+(v)=|E|.$$

In [ ]:
D = nx.DiGraph(name="Rede de seguidores")
D.add_edges_from([
    ("Ana", "Bruno"),
    ("Ana", "Carla"),
    ("Bruno", "Carla"),
    ("Carla", "Ana"),
    ("Carla", "Diego"),
    ("Diego", "Carla"),
    ("Eva", "Diego"),
])

pos_D = nx.spring_layout(D, seed=SEED)
desenhar_grafo(D, pos=pos_D, titulo="Rede direcionada de seguidores")

graus_direcionados = pd.DataFrame({
    "entrada": dict(D.in_degree()),
    "saída": dict(D.out_degree()),
    "total": dict(D.degree()),
})
display(graus_direcionados)

assert D.has_edge("Ana", "Bruno")
assert not D.has_edge("Bruno", "Ana")
assert sum(dict(D.in_degree()).values()) == D.number_of_edges()
assert sum(dict(D.out_degree()).values()) == D.number_of_edges()

### Conectividade forte e fraca

- Um dígrafo é **fortemente conectado** quando cada vértice alcança todos os outros respeitando as direções.
- É **fracamente conectado** quando o grafo se torna conectado ao ignorarmos as direções.

In [ ]:
print("Fracamente conectado?", nx.is_weakly_connected(D))
print("Fortemente conectado?", nx.is_strongly_connected(D))
print("Componentes fortemente conexos:", list(nx.strongly_connected_components(D)))

print("Sucessores de Carla:", list(D.successors("Carla")))
print("Predecessores de Carla:", list(D.predecessors("Carla")))

## 8. Laços, multigrafos e conversões

Um grafo simples não representa duas relações distintas entre o mesmo par sem perda de informação: uma nova inserção atualiza a mesma aresta. Para linhas de ônibus, voos ou interações de tipos diferentes, pode ser necessário um `MultiGraph` ou `MultiDiGraph`.

In [ ]:
MG = nx.MultiDiGraph(name="Trechos de transporte")
MG.add_edge("Natal", "Recife", empresa="Azul", duracao=65)
MG.add_edge("Natal", "Recife", empresa="Gol", duracao=70)
MG.add_edge("Recife", "Natal", empresa="Azul", duracao=68)
MG.add_edge("Natal", "Natal", empresa="Voo de treinamento", duracao=25)

print("Ordem:", MG.number_of_nodes())
print("Tamanho (arestas paralelas contam):", MG.number_of_edges())
print("Laços:", list(nx.selfloop_edges(MG, keys=True, data=True)))

for u, v, chave, dados in MG.edges(keys=True, data=True):
    print(f"{u} -> {v} | chave={chave} | {dados}")

### Cuidado com conversões

Converter um multigrafo em grafo simples pode descartar ou combinar relações de maneira inadequada. Antes de converter, defina a regra de agregação: menor duração, soma de capacidades, média de preços, contagem de eventos etc.

In [ ]:
# Agregação explícita: manter a menor duração por par direcionado.
DG_min = nx.DiGraph()
for u, v, dados in MG.edges(data=True):
    duracao = dados["duracao"]
    if not DG_min.has_edge(u, v) or duracao < DG_min[u][v]["duracao"]:
        DG_min.add_edge(u, v, duracao=duracao)

print(list(DG_min.edges(data=True)))

## 09. Decisões de modelagem

Antes de programar, responda:

1. **Quais são os vértices?** Pessoas, locais, estados, documentos ou componentes?
2. **O que uma aresta significa?** A relação é direcionada?
3. **Há múltiplas relações entre o mesmo par?** Se sim, elas devem ser agregadas?
4. **Há pesos?** Peso representa custo, capacidade, força, probabilidade ou distância?
5. **O tempo importa?** Uma sequência de fotografias estáticas é suficiente ou é preciso um grafo temporal?
6. **O que ficou de fora?** Toda abstração perde informação; documente a perda.

### Contraexemplo útil

Modelar “Ana segue Bruno” como uma aresta não direcionada transforma uma relação assimétrica em simétrica. O algoritmo pode executar sem erros e, ainda assim, responder à pergunta errada.

## 10. Prática guiada

### Exercício 1 — folhas

Em árvores, um vértice de grau 1 é chamado de **folha**. Implemente `obter_folhas(G)`.

**Contrato:** retornar uma lista com os vértices de grau 1. Um vértice isolado tem grau 0 e não é folha por esta definição.

In [ ]:
def obter_folhas(G):
    return [no for no, grau in G.degree() if grau == 1]

In [ ]:
T = nx.Graph()
T.add_edges_from([("a", "b"), ("a", "d"), ("c", "d")])
T.add_node("isolado")

assert set(obter_folhas(T)) == {"b", "c"}
print("Teste do Exercício 1 aprovado.")

### Exercício 2 — maior grau com empates

Implemente `vertices_maior_grau(G)`, retornando uma tupla `(lista_de_vertices, grau_maximo)`. Preserve todos os empates e ordene a lista pelo texto de cada rótulo.

In [ ]:
def vertices_maior_grau(G):
    if G.number_of_nodes() == 0:
        return ([], 0)
    grau_maximo = max(dict(G.degree()).values())
    vertices = sorted(
        (no for no, grau in G.degree() if grau == grau_maximo),
        key=str,
    )
    return (vertices, grau_maximo)

In [ ]:
assert vertices_maior_grau(nx.path_graph(4)) == ([1, 2], 2)
assert vertices_maior_grau(nx.Graph()) == ([], 0)
print("Teste do Exercício 2 aprovado.")

### Exercício 3 — vizinhos em comum

Implemente `vizinhos_em_comum(G, u, v)` usando operações de conjuntos. Retorne uma lista ordenada.

In [ ]:
def vizinhos_em_comum(G, u, v):
    comuns = set(G.neighbors(u)) & set(G.neighbors(v))
    return sorted(comuns, key=str)

In [ ]:
assert vizinhos_em_comum(SG, "Ana", "Carla") == ["Bruno", "Fábio"]
assert vizinhos_em_comum(SG, "Ana", "Gabi") == ["Fábio"]
print("Teste do Exercício 3 aprovado.")

### Exercício 4 — balanço em um dígrafo

Defina o balanço de um vértice como $b(v)=d^-(v)-d^+(v)$. Implemente `balanco_direcionado(D)` e retorne um dicionário. Ao final, explique por que a soma dos balanços deve ser zero.

In [ ]:
def balanco_direcionado(D):
    if not D.is_directed():
        raise nx.NetworkXError("A função requer um grafo direcionado.")
    return {
        no: D.in_degree(no) - D.out_degree(no)
        for no in D.nodes
    }

In [ ]:
balancos = balanco_direcionado(D)
print(balancos)
assert sum(balancos.values()) == 0
print("Teste do Exercício 4 aprovado.")

### Exercício 5 — desafio de modelagem

Uma universidade deseja representar pré-requisitos de disciplinas.

1. Qual classe de grafo deve ser usada?
2. Qual deve ser a direção de uma aresta?
3. O que significa um ciclo nesse contexto?
4. Qual algoritmo/propriedade ajudaria a produzir uma ordem válida de disciplinas?

Use o pequeno conjunto abaixo e verifique a resposta por código.

In [ ]:
prerequisitos = nx.DiGraph()
prerequisitos.add_edges_from([
    ("Programação I", "Estruturas de Dados I"),
    ("Estruturas de Dados I", "Estruturas de Dados II"),
    ("Matemática Discreta", "Estruturas de Dados II"),
    ("Estruturas de Dados II", "Projeto de Algoritmos"),
])

print("É um DAG?", nx.is_directed_acyclic_graph(prerequisitos))
print("Uma ordem topológica válida:")
print(list(nx.topological_sort(prerequisitos)))

## 11. Checklist de validação e erros frequentes

Antes de confiar na análise:

- [ ] a direção das arestas representa corretamente a relação?
- [ ] o significado e a unidade dos pesos estão documentados?
- [ ] laços e arestas paralelas foram tratados conscientemente?
- [ ] vértices isolados foram preservados na importação?
- [ ] o grafo deveria ser conectado? Se não for, isso é dado ou erro?
- [ ] os rótulos são únicos e estáveis?
- [ ] o desenho usa um layout reprodutível (`seed`) e não está sendo confundido com a estrutura?
- [ ] a fórmula/algoritmo escolhido aceita o tipo de grafo e os pesos usados?

### Erros frequentes em NetworkX

1. esquecer que métodos de modificação alteram o objeto original;
2. comparar `G.neighbors(v)` diretamente com uma lista sem converter o iterador;
3. usar `neighbors` em um `DiGraph` esperando predecessores e sucessores — ele retorna sucessores;
4. usar um caminho mínimo não ponderado quando o problema possui custos;
5. converter um multigrafo sem definir como agregar arestas paralelas;
6. interpretar proximidade visual como adjacência.

## 12. Síntese

Nesta aula, partimos da abstração $G=(V,E,w)$ e examinamos como decisões de domínio determinam o tipo de grafo e o significado de suas medidas. Usamos NetworkX para construir e consultar grafos, comparamos representações, verificamos identidades de grau, analisamos conectividade e caminhos, e discutimos dígrafos, pesos e multigrafos.

O estudo de caso mostrou uma ideia central de Estruturas de Dados II: representar estados como vértices e transições válidas como arestas transforma um espaço combinatório em um problema de grafos.

### Leituras e prática

- [Documentação oficial do NetworkX](https://networkx.org/documentation/stable/)
- [Tutorial oficial do NetworkX](https://networkx.org/documentation/stable/tutorial.html)
- [nx-guides](https://networkx.org/nx-guides/)
- Barabási, *Network Science* — leitura aberta em [networksciencebook.com](http://networksciencebook.com/)
- Michele Coscia, *The Atlas for the Aspiring Network Scientist* — [networkatlas.eu](https://www.networkatlas.eu/)
